# 16. Classes and Objects

At this point we have defined classes and created objects that represent the time of day and the day of the year. And we’ve defined methods that create, modify, and perform computations with these objects.

In this chapter we’ll continue our tour of object-oriented programming (OOP) by defining classes that represent geometric objects, including points, lines, rectangles, and circles. We’ll write methods that create and modify these objects, and we’ll use the jupyturtle module to draw them.

I’ll use these classes to demonstrate OOP topics including object identity and equivalence, shallow and deep copying, and polymorphism.

## 16.1. Creating a Point
In computer graphics a location on the screen is often represented using a pair of coordinates in an x-y plane. By convention, the point (0, 0) usually represents the upper-left corner of the screen, and (x, y) represents the point x units to the right and y units down from the origin. Compared to the Cartesian coordinate system you might have seen in a math class, the y axis is upside-down.

There are several ways we might represent a point in Python:

We can store the coordinates separately in two variables, x and y.

We can store the coordinates as elements in a list or tuple.

We can create a new type to represent points as objects.

In object-oriented programming, it would be most idiomatic to create a new type. To do that, we’ll start with a class definition for Point.

In [1]:
# thinkpython.py content
import contextlib
import io
import re

def extract_function_name(text):
    """Find a function definition and return its name.
     text: String
     returns: String or None
     """
    pattern = r"def\s+(\w+)\s*\("
    match = re.search(pattern, text)
    if match:
        func_name = match.group(1)
        return func_name
    else:
        return None

# the functions that define cell magic commands are only defined
# if we're running in Jupyter.
try:
    from IPython.core.magic import register_cell_magic
    from IPython.core.magic_arguments import argument, magic_arguments, parse_argstring

    @register_cell_magic
    def add_method_to(args, cell):

        # get the name of the function defined in this cell
        func_name = extract_function_name(cell)
        if func_name is None:
            return f"This cell doesn't define any new functions."

        # get the class we're adding it to
        namespace = get_ipython().user_ns
        class_name = args
        cls = namespace.get(class_name, None)
        if cls is None:
            return f"Class '{class_name}' not found."

        # save the old version of the function if it was already defined
        old_func = namespace.get(func_name, None)
        if old_func is not None:
            del namespace[func_name]

        # Execute the cell to define the function
        get_ipython().run_cell(cell)

        # get the newly defined function
        new_func = namespace.get(func_name, None)
        if new_func is None:
            return f"This cell didn't define {func_name}."

        # add the function to the class and remove it from the namespace
        setattr(cls, func_name, new_func)
        del namespace[func_name]

        # restore the old function to the namespace
        if old_func is not None:
            namespace[func_name] = old_func

    @register_cell_magic
    def expect_error(line, cell):
        try:
            get_ipython().run_cell(cell)
        except Exception as e:
            get_ipython().run_cell("%tb")

    @magic_arguments()
    @argument("exception", help="Type of exception to catch")
    @register_cell_magic
    def expect(line, cell):
        args = parse_argstring(expect, line)
        exception = eval(args.exception)
        try:
            get_ipython().run_cell(cell)
        except exception as e:
            get_ipython().run_cell("%tb")

    def traceback(mode):
        """Set the traceback mode.
         mode: string
         """
        with contextlib.redirect_stdout(io.StringIO()):
            get_ipython().run_cell(f"%xmode {mode}")

    traceback("Minimal")
except (ImportError, NameError):
    print("Warning: IPython is not available, cell magic not defined.")


In [2]:
%lsmagic

'Available line magics:\n%alias  %alias_magic  %autoawait  %autocall  %automagic  %autosave  %bookmark  %cat  %cd  %clear  %code_wrap  %colors  %conda  %config  %connect_info  %cp  %debug  %dhist  %dirs  %doctest_mode  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %lf  %lk  %ll  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %ls  %lsmagic  %lx  %macro  %magic  %mamba  %man  %matplotlib  %micromamba  %mkdir  %more  %mv  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %pip  %popd  %pprint  %precision  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %rep  %rerun  %reset  %reset_selective  %rm  %rmdir  %run  %save  %sc  %set_env  %store  %subshell  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %uv  %who  %who_ls  %whos  %xdel  %xmode\n\nAvailable cell magics:\n%%!  %%HTML  %%SVG  %%add_method_to  %%bash  %%capture  %%code_wrap  %%debug 

In [3]:
class Point:
    """Represents a point in a 2-D space."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f'Point({self.x}, {self.y})'

In [4]:
start = Point(0, 0)
print(start)

Point(0, 0)


In general, programmer-defined types are mutable, so we can write a method like translate that takes two numbers, dx and dy, and adds them to the attributes x and y.

In [5]:
%%add_method_to Point
def translate(self, dx, dy):
    self.x += dx
    self.y += dy

In [6]:
from copy import copy
end1 = copy(start)
end1.translate(300, 0)
print(end1)

Point(300, 0)


We can encapsulate those steps in another method called translated.

In the same way that the 'sort' method modifies a list and the 'sorted' function creates a new list, now we have a translate method that modifies a Point and a translated method that creates a new one.

In [7]:
%%add_method_to Point
def translated(self, dx, dy):
    point = copy(self)
    point.translate(dx, dy)
    return point

In [8]:
end2 = start.translated(0, 150)
print(end2)

Point(0, 150)


## 16.2. Creating a Line
Now let’s define a class that represents the line segment between two points. As usual, we’ll start with an __init__ method and a __str__ method.

In [9]:
class Line:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2

    def __str__(self):
        return f'Line({self.p1}, {self.p2})'

In [10]:
line1 = Line(start, end1)
print(line1)

Line(Point(0, 0), Point(300, 0))


In [11]:
import jupyturtle
print(jupyturtle.__file__)

/Users/shuyu/PycharmProjects/python-study-notes/.venv/lib/python3.12/site-packages/jupyturtle/__init__.py


In [12]:
import jupyturtle
print(jupyturtle.__file__)
print(jupyturtle.__doc__)          # should say "version 2024.04.01"
print('jumpto' in dir(jupyturtle)) # True?
print('moveto' in dir(jupyturtle)) # True?

/Users/shuyu/PycharmProjects/python-study-notes/.venv/lib/python3.12/site-packages/jupyturtle/__init__.py

🐢 jupyturtle.py
version 2024.04.01

True
True


In [13]:
# make a turtle from now on
from jupyturtle import make_turtle, jumpto, moveto

In [14]:
%%add_method_to Line
def draw(self):
    jumpto(self.p1.x, self.p1.y)
    moveto(self.p2.x, self.p2.y)

In [15]:
line2 = Line(start, end2)
print(line2)

Line(Point(0, 0), Point(0, 150))


In [16]:
make_turtle()
line1.draw()
line2.draw()

## 16.3. Equivalence and identity
Suppose we create two points with the same coordinates.

In [17]:
p1 = Point(200, 100)
p2 = Point(200, 100)

In [18]:
p1 == p2

False

If we want to change that behavior, we can provide a special method called __eq__ that defines what it means for two Point objects to be equal.

In [19]:
%%add_method_to Point
def __eq__(self, other):
    return self.x == other.x and self.y == other.y

In [20]:
p1 == p2

True

But the is operator still indicates that they are different objects.

It’s __not__ possible to override the is operator – it always checks whether the objects are identical. But for programmer-defined types, you can override the == operator so it checks whether the objects are equivalent. And you can define what equivalent means.

In [21]:
p1 is p2

False

## 16.4. Creating a Rectangle
Now let’s define a class that represents and draws rectangles. To keep things simple, we’ll assume that the rectangles are either vertical or horizontal, not at an angle. What attributes do you think we should use to specify the location and size of a rectangle?

There are at least two possibilities:

You could specify the width and height of the rectangle and the location of one corner.

You could specify two opposing corners.

At this point it’s hard to say whether either is better than the other, so let’s implement the first one. Here is the class definition.

In [22]:
class Rectangle:
    """Represents a rectangle.
    attributes: width, height, corner"""

    def __init__(self, width, height, corner):
        self.width = width
        self.height = height
        self.corner = corner

    def __str__(self):
        return f'Rectangle({self.width}, {self.height}, {self.corner})'

In [23]:
corner = Point(30, 20)
box1 = Rectangle(100, 50, corner)
print(box1)

Rectangle(100, 50, Point(30, 20))


In [24]:
%%add_method_to Rectangle
def make_points(self):
    p1 = self.corner
    p2 = p1.translated(self.width, 0)
    p3 = p2.translated(0, self.height)
    p4 = p3.translated(-self.width, 0)
    return p1, p2, p3, p4

In [25]:
%%add_method_to Rectangle
def make_lines(self):
    p1, p2, p3, p4 = self.make_points()
    return Line(p1, p2), Line(p2, p3), Line(p3, p4), Line(p4, p1)

In [26]:
%%add_method_to Rectangle
def draw(self):
    lines = self.make_lines()
    for line in lines:
        line.draw()

In [27]:
# here is an example
make_turtle()
line1.draw()
line2.draw()
box1.draw()

## 16.5. Changing rectangles
Now let’s consider two methods that modify rectangles, grow and translate. We’ll see that grow works as expected, but translate has a subtle bug. See if you can figure it out before I explain.

grow takes two numbers, dwidth and dheight, and adds them to the width and height attributes of the rectangle.

In [28]:
%%add_method_to Rectangle
def grow(self, dwidth, dheight):
    self.width += dwidth
    self.height += dheight

In [29]:
box2 = copy(box1)
box2.grow(60, 40)
print(box2)

Rectangle(160, 90, Point(30, 20))


In [30]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

In [31]:
%%add_method_to Rectangle
def translate(self, dx, dy):
    self.corner.translate(dx, dy)

In [32]:
box2.translate(30, 20)
print(box2)

Rectangle(160, 90, Point(60, 40))


In [33]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

It looks like both rectangles moved, which is not what we intended! The next section explains what went wrong.

## 16.6. Deep copy
When we use copy to duplicate box1, it copies the Rectangle object but not the Point object it contains. So box1 and box2 are different objects, as intended.

In [34]:
box1 is box2

False

In [35]:
box1.corner is box2.corner

True

What copy does is called a shallow copy because it copies the object but not the objects it contains. As a result, changing the width or height of one Rectangle does not affect the other, but changing the attributes of the shared Point affects both! This behavior is confusing and error-prone.

Fortunately, the copy module provides another function, called deepcopy, that copies not only the object but also the objects it refers to, and the objects they refer to, and so on. This operation is called a deep copy.

To demonstrate, let’s start with a new Rectangle that contains a new Point.

In [36]:
corner = Point(20, 20)
box3 = Rectangle(100, 50, corner)
print(box3)

Rectangle(100, 50, Point(20, 20))


In [37]:
from copy import deepcopy

box4 = deepcopy(box3)

In [38]:
box3.corner is box4.corner

False

In [39]:
box3.translate(50, 30)
box4.grow(100, 60)

In [40]:
make_turtle()
line1.draw()
line2.draw()
box3.draw()
box4.draw()

## 16.7. Polymorphism
In the previous example, we invoked the draw method on two Line objects and two Rectangle objects. We can do the same thing more concisely by making a list of objects.

In [41]:
shapes = [line1, line2, box3, box4]

In [42]:
make_turtle()

for shape in shapes:
    shape.draw()

The first and second time through the loop, shape refers to a Line object, so when draw is invoked, the method that runs is the one defined in the Line class.

The third and fourth time through the loop, shape refers to a Rectangle object, so when draw is invoked, the method that runs is the one defined in the Rectangle class.

In a sense, each object knows how to draw itself. This feature is called polymorphism. The word comes from Greek roots that mean “many shaped”. In object-oriented programming, polymorphism is the ability of different types to provide the same methods, which makes it possible to perform many computations – like drawing shapes – by invoking the same method on different types of objects.

As an exercise at the end of this chapter, you’ll define a new class that represents a circle and provides a draw method. Then you can use polymorphism to draw lines, rectangles, and circles.



## 16.8. Debugging
In this chapter, we ran into a subtle bug that happened because we created a Point that was shared by two Rectangle objects, and then we modified the Point. In general, there are two ways to avoid problems like this: you can avoid sharing objects or you can avoid modifying them.

To avoid sharing objects, you can use deep copy, as we did in this chapter.

To avoid modifying objects, consider replacing impure functions like translate with pure functions like translated. For example, here’s a version of translated that creates a new Point and never modifies its attributes.

In [ ]:
    def translated(self, dx=0, dy=0):
        x = self.x + dx
        y = self.y + dy
        return Point(x, y)

Python provides features that make it easier to avoid modifying objects. They are beyond the scope of this book, but if you are curious, ask a virtual assistant, “How do I make a Python object immutable?”

Creating a new object takes more time than modifying an existing one, but the difference seldom matters in practice. Programs that avoid shared objects and impure functions are often easier to develop, test, and debug – and the best kind of debugging is the kind you don’t have to do.

## 16.9. Glossary
shallow copy: A copy operation that does not copy nested objects.

deep copy: A copy operation that also copies nested objects.

polymorphism: The ability of a method or operator to work with multiple types of objects.

## 16.10. Exercises

In [43]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


### 16.10.1. Ask a virtual assistant
For all of the following exercises, consider asking a virtual assistant for help. If you do, you’ll want include as part of the prompt the class definitions for Point, Line, and Rectangle – otherwise the VA will make a guess about their attributes and functions, and the code it generates won’t work.

### 16.10.2. Exercise
Write an __eq__ method for the Line class that returns True if the Line objects refer to Point objects that are equivalent, in either order.

You can use the following outline to get started.

In [44]:
%%add_method_to Line

def __eq__(self, other):
    return sorted([self.p1.x, self.p1.y, self.p2.x, self.p2.y]) == sorted([other.p1.x, self.p1.y, other.p2.x, self.p2.y])

In [45]:
start1 = Point(0, 0)
start2 = Point(0, 0)
end = Point(200, 100)

In [46]:
line_a = Line(start1, end)
line_b = Line(start2, end)
line_a == line_b    # should be True

True

In [47]:
line_c = Line(end, start1)
line_a == line_c     # should be True

True

In [48]:
line_b == line_c     # should be True

True

In [49]:
line_d = Line(start1, start2)
line_a == line_d    # should be False

False

### 16.10.3. Exercise
Write a Line method called midpoint that computes the midpoint of a line segment and returns the result as a Point object.

You can use the following outline to get started.

In [70]:
%%add_method_to Line

def midpoint(self):
    x = (self.p1.x + self.p2.x)/2
    y = (self.p1.y + self.p2.y)/2
    return Point(x, y)

In [71]:
start = Point(0, 0)
end1 = Point(300, 0)
end2 = Point(0, 150)
line1 = Line(start, end1)
line2 = Line(start, end2)

In [72]:
mid1 = line1.midpoint()
print(mid1)

Point(150.0, 0.0)


In [73]:
mid2 = line2.midpoint()
print(mid2)

Point(0.0, 75.0)


In [74]:
line3 = Line(mid1, mid2)

In [75]:
make_turtle()

for shape in [line1, line2, line3]:
    shape.draw()

### 16.10.4. Exercise
Write a Rectangle method called midpoint that find the point in the center of a rectangle and returns the result as a Point object.

You can use the following outline to get started.

In [76]:
%%add_method_to Rectangle

def midpoint(self):
    x = self.corner.x + self.width/2
    y = self.corner.y + self.height/2
    return Point(x, y)

In [77]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [78]:
mid = rectangle.midpoint()
print(mid)

Point(80.0, 60.0)


In [79]:
diagonal = Line(corner, mid)

In [80]:
make_turtle()

for shape in [line1, line2, rectangle, diagonal]:
    shape.draw()

### 16.10.5. Exercise
Write a Rectangle method called make_cross that:

Uses make_lines to get a list of Line objects that represent the four sides of the rectangle.

Computes the midpoints of the four lines.

Makes and returns a list of two Line objects that represent lines connecting opposite midpoints, forming a cross through the middle of the rectangle.

You can use this outline to get started.

In [86]:
%%add_method_to Rectangle

def make_cross(self):
    lines = self.make_lines()  # four Line objects in the clock-wise order
    midpoints = []
    for line in lines:
        midpoints.append(line.midpoint())
    line1 = Line(midpoints[0], midpoints[2])
    line2 = Line(midpoints[1], midpoints[3])
    return [line1, line2]

In [87]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [88]:
lines = rectangle.make_cross()

In [89]:
make_turtle()

rectangle.draw()
for line in lines:
    line.draw()

### 16.10.6. Exercise
Write a definition for a class named Circle with attributes center and radius, where center is a Point object and radius is a number. Include special methods __init__ and a __str__, and a method called draw that uses jupyturtle functions to draw the circle.

You can use the following function, which is a version of the circle function we wrote in Chapter 4.

In [106]:
from jupyturtle import make_turtle, forward, left, right
import math

def draw_circle(radius):
    circumference = 2 * math.pi * radius
    n = 30
    length = circumference / n
    angle = 360 / n
    left(angle / 2)
    for i in range(n):
        forward(length)
        left(angle)

class Circle:
    """Represents a circle.
    attributes:
    center: a Point object,
    radius: a number.
    methods: init, str, draw"""

    def __init__(self, center, radius):
        self.center = center
        self.radius = radius

    def __str__(self):
        return f'Circle({self.center}, {self.radius})'

    def draw(self):
        jumpto(self.center.x, self.center.y +  radius)
        draw_circle(self.radius)

In [107]:
corner = Point(20, 20)
rectangle = Rectangle(100, 100, corner)

In [108]:
center = rectangle.midpoint()
radius = rectangle.height / 2

circle = Circle(center, radius)
print(circle)

Circle(Point(70.0, 70.0), 50.0)


In [109]:
make_turtle(delay=0.01)

rectangle.draw()
circle.draw()